<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/DenseNetMimarisi/DenseNet121Mimarisi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict

class _DenseLayer(nn.Module):
    """DenseNet'in temel building block'u - dense connectivity ile feature concatenation"""

    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        super(_DenseLayer, self).__init__()

        # Bottleneck design: 1x1 conv -> 3x3 conv
        self.add_module('norm1', nn.BatchNorm2d(num_input_features))
        self.add_module('relu1', nn.ReLU(inplace=True))
        self.add_module('conv1', nn.Conv2d(num_input_features, bn_size * growth_rate,
                                         kernel_size=1, stride=1, bias=False))

        self.add_module('norm2', nn.BatchNorm2d(bn_size * growth_rate))
        self.add_module('relu2', nn.ReLU(inplace=True))
        self.add_module('conv2', nn.Conv2d(bn_size * growth_rate, growth_rate,
                                         kernel_size=3, stride=1, padding=1, bias=False))

        self.drop_rate = drop_rate

    def forward(self, x):
        """Dense connectivity: concatenate input with previous features"""
        if isinstance(x, torch.Tensor):
            prev_features = [x]
        else:
            prev_features = x

        # Tüm önceki feature'ları concatenate et
        concated_features = torch.cat(prev_features, 1)

        # Bottleneck transformation
        bottleneck_output = self.conv1(self.relu1(self.norm1(concated_features)))
        new_features = self.conv2(self.relu2(self.norm2(bottleneck_output)))

        # Dropout uygulaması
        if self.drop_rate > 0:
            new_features = F.dropout(new_features, p=self.drop_rate, training=self.training)

        return new_features


class _DenseBlock(nn.Module):
    """Dense Block: Dense layer'ların sequential arrangement'ı"""

    def __init__(self, num_layers, num_input_features, bn_size, growth_rate, drop_rate):
        super(_DenseBlock, self).__init__()

        for i in range(num_layers):
            layer = _DenseLayer(
                num_input_features + i * growth_rate,
                growth_rate=growth_rate,
                bn_size=bn_size,
                drop_rate=drop_rate
            )
            self.add_module('denselayer%d' % (i + 1), layer)

    def forward(self, init_features):
        """Forward pass with feature concatenation"""
        features = [init_features]

        for name, layer in self.named_children():
            new_features = layer(features)
            features.append(new_features)

        return torch.cat(features, 1)


class _Transition(nn.Module):
    """Transition Layer: Dimensionality reduction between dense blocks"""

    def __init__(self, num_input_features, num_output_features):
        super(_Transition, self).__init__()

        self.add_module('norm', nn.BatchNorm2d(num_input_features))
        self.add_module('relu', nn.ReLU(inplace=True))
        self.add_module('conv', nn.Conv2d(num_input_features, num_output_features,
                                        kernel_size=1, stride=1, bias=False))
        self.add_module('pool', nn.AvgPool2d(kernel_size=2, stride=2))

    def forward(self, x):
        return self.pool(self.conv(self.relu(self.norm(x))))


class DenseNet(nn.Module):
    """
    DenseNet-BC (Bottleneck + Compression) Implementation

    Args:
        growth_rate (int): Her layer'ın ürettiği feature map sayısı (k)
        block_config (list): Her dense block'taki layer sayıları
        num_init_features (int): İlk convolution'dan sonraki feature sayısı
        bn_size (int): Bottleneck size multiplier
        drop_rate (float): Dropout probability
        num_classes (int): Classification class sayısı
        compression_factor (float): Feature compression oranı (θ)
    """

    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_init_features=64, bn_size=4, drop_rate=0,
                 num_classes=1000, compression_factor=0.5):

        super(DenseNet, self).__init__()

        # İlk convolution ve pooling layer'ları
        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, num_init_features, kernel_size=7, stride=2, padding=3, bias=False)),
            ('norm0', nn.BatchNorm2d(num_init_features)),
            ('relu0', nn.ReLU(inplace=True)),
            ('pool0', nn.MaxPool2d(kernel_size=3, stride=2, padding=1)),
        ]))

        # Dense blocks ve transition layers
        num_features = num_init_features
        for i, num_layers in enumerate(block_config):
            # Dense block ekleme
            block = _DenseBlock(
                num_layers=num_layers,
                num_input_features=num_features,
                bn_size=bn_size,
                growth_rate=growth_rate,
                drop_rate=drop_rate
            )
            self.features.add_module('denseblock%d' % (i + 1), block)
            num_features = num_features + num_layers * growth_rate

            # Transition layer (son block hariç)
            if i != len(block_config) - 1:
                trans = _Transition(
                    num_input_features=num_features,
                    num_output_features=int(num_features * compression_factor)
                )
                self.features.add_module('transition%d' % (i + 1), trans)
                num_features = int(num_features * compression_factor)

        # Final batch normalization
        self.features.add_module('norm5', nn.BatchNorm2d(num_features))

        # Linear classifier
        self.classifier = nn.Linear(num_features, num_classes)

        # Weight initialization
        self._initialize_weights()

    def forward(self, x):
        """Forward propagation through DenseNet"""
        features = self.features(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        out = self.classifier(out)
        return out

    def _initialize_weights(self):
        """Xavier/He weight initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)


def densenet121(pretrained=False, **kwargs):
    """DenseNet-121 configuration"""
    model = DenseNet(growth_rate=32, block_config=(6, 12, 24, 16), **kwargs)
    return model

def densenet169(pretrained=False, **kwargs):
    """DenseNet-169 configuration"""
    model = DenseNet(growth_rate=32, block_config=(6, 12, 32, 32), **kwargs)
    return model

def densenet201(pretrained=False, **kwargs):
    """DenseNet-201 configuration"""
    model = DenseNet(growth_rate=32, block_config=(6, 12, 48, 32), **kwargs)
    return model


# Kullanım örneği ve model analizi
if __name__ == "__main__":
    # Model instance oluşturma
    model = densenet121(num_classes=10)  # CIFAR-10 için

    # Model parameter sayısını hesaplama
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"DenseNet-121 Model Statistics:")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

    # Dummy input ile test
    dummy_input = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        output = model(dummy_input)
        print(f"Input shape: {dummy_input.shape}")
        print(f"Output shape: {output.shape}")

    # Feature extraction için hook function
    def hook_fn(module, input, output):
        print(f"Layer: {module.__class__.__name__}, Output shape: {output.shape}")

    # İlk dense block'a hook ekleme (feature evolution'ı görmek için)
    handle = model.features.denseblock1.register_forward_hook(hook_fn)

    # Model evaluation
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)

    handle.remove()  # Hook'u temizle

DenseNet-121 Model Statistics:
Total Parameters: 6,964,106
Trainable Parameters: 6,964,106
Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 10])
Layer: _DenseBlock, Output shape: torch.Size([1, 256, 56, 56])
